# Silver Validation Notebook (Workshop 2)

This notebook validates the Silver layer outputs generated by the Airflow DAG.
It reads the three Parquet datasets from local disk and runs lightweight quality checks.

**Datasets validated:**
- `silver_titles` — normalised title-level metadata merged from TMDB and Rotten Tomatoes
- `silver_rt_reviews` — critic and audience review text extracted from Rotten Tomatoes
- `silver_tmdb_reviews` — user review text extracted from the TMDB reviews endpoint

> **Note:** Run the first cell to install dependencies if `pyarrow` is not available in your local environment.

In [1]:
# Install required dependencies if not already present
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow', 'pandas'])

0

In [2]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)

In [3]:
# Resolve the silver layer directory relative to this notebook.
# The notebook lives in notebooks/, so ../datalake_silver points to the project root.
silver_dir = Path('../datalake_silver').resolve()
assert silver_dir.exists(), f"Silver directory not found: {silver_dir}"
silver_dir

WindowsPath('C:/Users/juanp/OneDrive/Documentos/Trabajos u/Analisisdatos/Streaming_Data_Analysis/datalake_silver')

In [4]:
def latest_parquet(prefix: str, base_dir: Path) -> Path:
    """Return the most recent Parquet file matching a given prefix."""
    files = sorted(base_dir.glob(f'{prefix}_*.parquet'))
    if not files:
        raise FileNotFoundError(f'No Parquet files found for prefix "{prefix}" in {base_dir}')
    return files[-1]

titles_path      = latest_parquet('silver_titles',       silver_dir)
rt_reviews_path  = latest_parquet('silver_rt_reviews',   silver_dir)
tmdb_reviews_path= latest_parquet('silver_tmdb_reviews', silver_dir)

print('Files selected:')
for p in [titles_path, rt_reviews_path, tmdb_reviews_path]:
    print(' ', p.name)

Files selected:
  silver_titles_20260420_000300.parquet
  silver_rt_reviews_20260420_000300.parquet
  silver_tmdb_reviews_20260420_000300.parquet


In [5]:
df_titles       = pd.read_parquet(titles_path,       engine='pyarrow')
df_rt_reviews   = pd.read_parquet(rt_reviews_path,   engine='pyarrow')
df_tmdb_reviews = pd.read_parquet(tmdb_reviews_path, engine='pyarrow')

print('Loaded datasets:')
print(f'  silver_titles       : {df_titles.shape}')
print(f'  silver_rt_reviews   : {df_rt_reviews.shape}')
print(f'  silver_tmdb_reviews : {df_tmdb_reviews.shape}')

Loaded datasets:
  silver_titles       : (98, 20)
  silver_rt_reviews   : (80, 22)
  silver_tmdb_reviews : (11, 22)


## 1) Shape and Columns

In [6]:
datasets = {
    'silver_titles':       df_titles,
    'silver_rt_reviews':   df_rt_reviews,
    'silver_tmdb_reviews': df_tmdb_reviews,
}

for name, df in datasets.items():
    print('=' * 80)
    print(f'Dataset : {name}')
    print(f'Shape   : {df.shape}')
    print(f'Columns : {list(df.columns)}')

Dataset : silver_titles
Shape   : (98, 20)
Columns : ['canonical_title', 'source_title', 'source', 'source_type', 'source_id', 'url', 'poster_url', 'release_date', 'release_year', 'popularity', 'vote_average', 'vote_count', 'critics_score', 'audience_score', 'overview', 'genre_ids', 'scraped_at', 'bronze_file_name', 'bronze_source_type', 'processed_at']
Dataset : silver_rt_reviews
Shape   : (80, 22)
Columns : ['canonical_title', 'source', 'review_type', 'reviewer_name', 'publication', 'review_text', 'review_text_clean', 'sentiment_label', 'original_score', 'rating', 'is_verified', 'is_top_critic', 'review_date', 'source_url', 'publication_review_url', 'title_url', 'critics_score', 'audience_score', 'scraped_at', 'bronze_file_name', 'bronze_source_type', 'processed_at']
Dataset : silver_tmdb_reviews
Shape   : (11, 22)
Columns : ['movie_id', 'canonical_title', 'source', 'reviewer_name', 'review_text', 'review_text_clean', 'rating', 'review_date', 'review_id', 'review_url', 'popularity', 

## 2) Null Analysis

In [7]:
def null_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Return a DataFrame summarising null counts and percentages per column."""
    nulls = df.isnull().sum()
    pct   = (nulls / len(df) * 100).round(1)
    return (
        pd.DataFrame({'null_count': nulls, 'null_pct': pct})
        .query('null_count > 0')
        .sort_values('null_count', ascending=False)
        .rename_axis(f'{name} — column')
    )

for name, df in datasets.items():
    print(f'\n{"=" * 60}')
    print(f'Null report: {name}')
    report = null_report(df, name)
    if report.empty:
        print('  No null values found.')
    else:
        print(report.to_string())


Null report: silver_titles
                        null_count  null_pct
silver_titles — column                      
audience_score                  68      69.4
critics_score                   53      54.1
url                             48      49.0
poster_url                      48      49.0
overview                        29      29.6
source_id                       29      29.6
popularity                      29      29.6
vote_average                    29      29.6
genre_ids                       29      29.6
vote_count                      29      29.6
release_date                     8       8.2
release_year                     8       8.2

Null report: silver_rt_reviews
                            null_count  null_pct
silver_rt_reviews — column                      
original_score                      59      73.8
publication                         40      50.0
sentiment_label                     40      50.0
rating                              40      50.0
is_verified     

## 3) Duplicate Check

In [8]:
dedup_keys = {
    'silver_titles':       ['source', 'source_title', 'url', 'source_id'],
    'silver_rt_reviews':   ['canonical_title', 'review_type', 'reviewer_name', 'review_date', 'review_text'],
    'silver_tmdb_reviews': ['movie_id', 'review_id'],
}

results = []
for name, df in datasets.items():
    keys = dedup_keys[name]
    available_keys = [k for k in keys if k in df.columns]
    dups = df.duplicated(subset=available_keys).sum()
    results.append({'dataset': name, 'total_rows': len(df), 'duplicate_rows': dups, 'dedup_keys': str(available_keys)})

pd.DataFrame(results)

,dataset,total_rows,duplicate_rows,dedup_keys
0,silver_titles,98,0,"['source', 'source_title', 'url', 'source_id']"
1,silver_rt_reviews,80,0,"['canonical_title', 'review_type', 'reviewer_n..."
2,silver_tmdb_reviews,11,0,"['movie_id', 'review_id']"


## 4) Source Distribution Checks

In [9]:
print('=== silver_titles: source distribution ===')
display(df_titles['source'].value_counts(dropna=False).to_frame('rows'))

print('\n=== silver_rt_reviews: review_type distribution ===')
display(df_rt_reviews['review_type'].value_counts(dropna=False).to_frame('rows'))

print('\n=== silver_rt_reviews: bronze_source_type distribution ===')
display(df_rt_reviews['bronze_source_type'].value_counts(dropna=False).to_frame('rows'))

print('\n=== silver_tmdb_reviews: source distribution ===')
display(df_tmdb_reviews['source'].value_counts(dropna=False).to_frame('rows'))

=== silver_titles: source distribution ===


,rows
source,
tmdb,48
rottentomatoes,29
"rottentomatoes, tmdb",21



=== silver_rt_reviews: review_type distribution ===


,rows
review_type,
critic,40
audience,40



=== silver_rt_reviews: bronze_source_type distribution ===


,rows
bronze_source_type,
rt_reviews_theaters,80



=== silver_tmdb_reviews: source distribution ===


,rows
source,
tmdb,11


## 5) Validate review_text vs review_text_clean

In [ ]:
def text_clean_checks(df: pd.DataFrame, text_col: str = 'review_text', clean_col: str = 'review_text_clean') -> pd.DataFrame:
    """Compare raw and cleaned review text to confirm the cleaning pipeline ran."""
    total = len(df)
    both_present = df[text_col].notna() & df[clean_col].notna()
    changed = both_present & (df[text_col] != df[clean_col])

    return pd.DataFrame({
        'metric': [
            'total_rows',
            'raw_text_not_null',
            'clean_text_not_null',
            'both_present',
            'changed_after_cleaning',
        ],
        'value': [
            total,
            int(df[text_col].notna().sum()),
            int(df[clean_col].notna().sum()),
            int(both_present.sum()),
            int(changed.sum()),
        ],
    })

print('=== RT reviews — text cleaning stats ===')
display(text_clean_checks(df_rt_reviews))

print('\n=== TMDb reviews — text cleaning stats ===')
display(text_clean_checks(df_tmdb_reviews))

In [11]:
# Show sample rows where the cleaning pipeline actually changed content
rt_changed = df_rt_reviews[
    df_rt_reviews['review_text'].notna() &
    df_rt_reviews['review_text_clean'].notna() &
    (df_rt_reviews['review_text'] != df_rt_reviews['review_text_clean'])
]

tmdb_changed = df_tmdb_reviews[
    df_tmdb_reviews['review_text'].notna() &
    df_tmdb_reviews['review_text_clean'].notna() &
    (df_tmdb_reviews['review_text'] != df_tmdb_reviews['review_text_clean'])
]

print(f'RT rows changed by cleaning   : {len(rt_changed)}')
display(rt_changed[['canonical_title', 'review_text', 'review_text_clean']].head(3))

print(f'\nTMDb rows changed by cleaning : {len(tmdb_changed)}')
display(tmdb_changed[['canonical_title', 'review_text', 'review_text_clean']].head(3))

RT rows changed by cleaning   : 79


,canonical_title,review_text,review_text_clean
0,lee cronin's the mummy,"A cliché, dime-store exorcist movie.","a cliché, dime-store exorcist movie."
1,lee cronin's the mummy,"Still, with the resulting film, even the word ...","still, with the resulting film, even the word ..."
2,lee cronin's the mummy,While it goes hard on the cringe-inducing gore...,while it goes hard on the cringe-inducing gore...



TMDb rows changed by cleaning : 11


,canonical_title,review_text,review_text_clean
0,project hail mary,When times are tough and world-weary souls hav...,when times are tough and world-weary souls hav...
1,project hail mary,I think this might be my favourite sci-fi film...,i think this might be my favourite sci-fi film...
2,project hail mary,Incredible movie. So heartwarming and funny. T...,incredible movie. so heartwarming and funny. t...


## 6) Lineage Evidence Checks

In [12]:
def lineage_completeness(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Verify that every row can be traced back to its Bronze source file."""
    required = ['bronze_file_name', 'bronze_source_type', 'processed_at']
    rows = []
    for col in required:
        not_null = int(df[col].notna().sum()) if col in df.columns else 0
        rows.append({
            'dataset':      name,
            'column':       col,
            'not_null_rows': not_null,
            'total_rows':   len(df),
            'coverage_pct': round(not_null / len(df) * 100, 1) if len(df) > 0 else 0.0,
        })
    return pd.DataFrame(rows)

lineage_dfs = [
    lineage_completeness(df_titles,       'silver_titles'),
    lineage_completeness(df_rt_reviews,   'silver_rt_reviews'),
    lineage_completeness(df_tmdb_reviews, 'silver_tmdb_reviews'),
]

pd.concat(lineage_dfs, ignore_index=True)

,dataset,column,not_null_rows,total_rows,coverage_pct
0,silver_titles,bronze_file_name,98,98,100.0
1,silver_titles,bronze_source_type,98,98,100.0
2,silver_titles,processed_at,98,98,100.0
3,silver_rt_reviews,bronze_file_name,80,80,100.0
4,silver_rt_reviews,bronze_source_type,80,80,100.0
5,silver_rt_reviews,processed_at,80,80,100.0
6,silver_tmdb_reviews,bronze_file_name,11,11,100.0
7,silver_tmdb_reviews,bronze_source_type,11,11,100.0
8,silver_tmdb_reviews,processed_at,11,11,100.0


In [ ]:
# Show which Bronze files contributed to each Silver dataset
for name, df in datasets.items():
    if 'bronze_file_name' in df.columns:
        # bronze_file_name may contain comma-separated values when multiple files merged
        all_files = (
            df['bronze_file_name']
            .dropna()
            .str.split(',')
            .explode()
            .str.strip()
            .unique()
        )
        print(f'\n{name} — Bronze source files ({len(all_files)}):')
        for f in sorted(all_files):
            print(f'  {f}')

## 7) Compact Workshop 2 Summary

In [13]:
summary = pd.DataFrame([
    {
        'dataset':        'silver_titles',
        'rows':           len(df_titles),
        'cols':           df_titles.shape[1],
        'unique_sources': df_titles['source'].nunique(dropna=True),
        'null_pct_avg':   round(df_titles.isnull().mean().mean() * 100, 1),
        'parquet_file':   titles_path.name,
    },
    {
        'dataset':        'silver_rt_reviews',
        'rows':           len(df_rt_reviews),
        'cols':           df_rt_reviews.shape[1],
        'unique_sources': df_rt_reviews['source'].nunique(dropna=True),
        'null_pct_avg':   round(df_rt_reviews.isnull().mean().mean() * 100, 1),
        'parquet_file':   rt_reviews_path.name,
    },
    {
        'dataset':        'silver_tmdb_reviews',
        'rows':           len(df_tmdb_reviews),
        'cols':           df_tmdb_reviews.shape[1],
        'unique_sources': df_tmdb_reviews['source'].nunique(dropna=True),
        'null_pct_avg':   round(df_tmdb_reviews.isnull().mean().mean() * 100, 1),
        'parquet_file':   tmdb_reviews_path.name,
    },
])

summary

,dataset,rows,cols,unique_sources,null_pct_avg,parquet_file
0,silver_titles,98,20,3,20.8,silver_titles_20260420_000300.parquet
1,silver_rt_reviews,80,22,1,19.3,silver_rt_reviews_20260420_000300.parquet
2,silver_tmdb_reviews,11,22,1,0.4,silver_tmdb_reviews_20260420_000300.parquet
